# NLP SQL data generator — Google Colab

Install deps, load this project, seed sample DBs, configure the LLM in **section 7** (OpenAI **or** Ollama), then run queries.

**Get the code into Colab (pick one path):**

1. **Git clone:** Uncomment `REPO_URL` in section 1 and run it (push this repo to GitHub first if needed).
2. **Zip upload:** Set `RUN_ZIP_UPLOAD = True` in section 2, run that cell, and pick your zip of the whole project folder.

Then run the remaining cells in order.

## 1) Optional: clone from Git

Skip if you use the zip path in section 2.

In [ ]:
# Uncomment and set your repo URL:
# REPO_URL = "https://github.com/YOUR_USER/nlp-sql-data-generator.git"
# !git clone "$REPO_URL" /content/nlp-sql-data-generator

## 2) Optional: upload project zip

Leave `RUN_ZIP_UPLOAD = False` when you used **git clone** (default). Set it to `True` only when uploading a zip.

In [ ]:
import io, os, zipfile
from pathlib import Path

RUN_ZIP_UPLOAD = False  # True => pick a zip of the project from your computer

if not RUN_ZIP_UPLOAD:
    print("Skipping zip upload. Clone the repo (section 1) or set RUN_ZIP_UPLOAD=True.")
else:
    from google.colab import files

    PROJECT_DIR = Path("/content/nlp-sql-data-generator")
    uploaded = files.upload()
    for name, data in uploaded.items():
        zf = zipfile.ZipFile(io.BytesIO(data))
        zf.extractall("/content")
        zf.close()
        print("Extracted:", name)

    if not PROJECT_DIR.is_dir():
        candidates = [
            p
            for p in Path("/content").iterdir()
            if p.is_dir() and (p / "pyproject.toml").is_file()
        ]
        if len(candidates) == 1:
            PROJECT_DIR = candidates[0]
        else:
            raise FileNotFoundError(
                f"Expected project at {PROJECT_DIR}. Found pyproject.toml under: {candidates}. "
                "Set PROJECT_DIR manually."
            )

    os.chdir(PROJECT_DIR)
    print("PROJECT_DIR =", PROJECT_DIR.resolve())

## 3) Move into the project directory

Required after **git clone**. Safe to run after zip upload too.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/nlp-sql-data-generator")
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing {PROJECT_DIR}. Clone the repo (section 1) or upload a zip (section 2)."
    )

os.chdir(PROJECT_DIR)
print("cwd =", os.getcwd())

## 4) Install Python dependencies

In [ ]:
%pip install -q -r requirements.txt

## 5) Make `nlp_sql` importable

Adds `src/` to `sys.path` (no `pip install -e .` required in Colab).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
SRC = ROOT / "src"
if not (SRC / "nlp_sql").is_dir():
    raise FileNotFoundError(f"Missing {SRC / 'nlp_sql'} — cwd should be project root.")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nlp_sql

print("nlp_sql:", nlp_sql.__file__)

## 6) Create sample databases (`data/sales.db`, `data/hr.db`)

In [ ]:
!python scripts/seed_sample_dbs.py

## 7) LLM: OpenAI **or** Ollama

In the **next cell**, set `BACKEND`:

| Value | Use case | `OPENAI_API_KEY` |
|--------|-----------|------------------|
| `"openai"` | OpenAI (or another **cloud** OpenAI-compatible API) | Required: Colab **Secret** `OPENAI_API_KEY` or paste when asked |
| `"ollama_colab"` | **Ollama runs on this Colab VM** (script installs Ollama, starts `ollama serve`, runs `ollama pull`) | **Not used** — leave unset / empty |
| `"ollama_remote"` | Ollama runs on **your laptop** and you expose it (ngrok, Cloudflare Tunnel, etc.) with a URL that ends in **`/v1`** | Usually **not** needed |

**Important:** Colab cannot talk to `http://127.0.0.1:11434` on your home PC. For Ollama on your own machine, use **`ollama_remote`** plus a **public** base URL (tunnel), or use **`ollama_colab`** so Ollama runs inside Colab.

First `ollama_colab` run can take several minutes (installer + model download).

In [ ]:
# ============ LLM backend (pick one) ============
BACKEND = "ollama_colab"  # "openai" | "ollama_colab" | "ollama_remote"

# Ollama: model name (must exist after `ollama pull` — change if you prefer another tag)
OLLAMA_MODEL = "llama3.2"

# Only for BACKEND == "ollama_remote": your public OpenAI-compatible base URL including /v1
# Example: "https://abcd-12-34-56-78.ngrok-free.app/v1"
OLLAMA_BASE_URL = ""

import os
import shutil
import subprocess
import time
from pathlib import Path

import httpx
import yaml

CFG_PATH = Path("config.yaml")
cfg = yaml.safe_load(CFG_PATH.read_text())
cfg.setdefault("llm", {})

if BACKEND == "openai":
    key = None
    try:
        from google.colab import userdata

        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        key = None

    if not key:
        key = input("OPENAI_API_KEY (paste): ").strip()

    if not key:
        raise ValueError("OPENAI_API_KEY is required when BACKEND == 'openai'")

    os.environ["OPENAI_API_KEY"] = key
    cfg["llm"]["base_url"] = "https://api.openai.com/v1"
    cfg["llm"]["model"] = "gpt-4o-mini"

elif BACKEND == "ollama_colab":
    # Ollama on the Colab VM (not on your laptop).
    os.environ.pop("OPENAI_API_KEY", None)

    print("Installing Ollama (one-time per Colab runtime) …")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

    ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"

    print("Starting `ollama serve` in background …")
    log = open("ollama_serve.log", "w")
    subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print("Waiting for Ollama HTTP API …")
    for i in range(90):
        try:
            r = httpx.get("http://127.0.0.1:11434/api/tags", timeout=2.0)
            if r.status_code == 200:
                break
        except Exception:
            pass
        time.sleep(1)
        if i % 10 == 0:
            print(" … still waiting", i, "s")
    else:
        raise RuntimeError("Ollama did not become ready. Check ollama_serve.log in the file browser.")

    print(f"Pulling model `{OLLAMA_MODEL}` (first time can take several minutes) …")
    subprocess.run([ollama_bin, "pull", OLLAMA_MODEL], check=True)

    cfg["llm"]["base_url"] = "http://127.0.0.1:11434/v1"
    cfg["llm"]["model"] = OLLAMA_MODEL

elif BACKEND == "ollama_remote":
    os.environ.pop("OPENAI_API_KEY", None)
    base = (OLLAMA_BASE_URL or "").strip().rstrip("/")
    if not base:
        base = input(
            "Paste Ollama **OpenAI-compatible** base URL (must end with /v1), e.g. https://....ngrok-free.app/v1\n> "
        ).strip().rstrip("/")
    if not base.endswith("/v1"):
        base = f"{base}/v1"
    cfg["llm"]["base_url"] = base
    cfg["llm"]["model"] = OLLAMA_MODEL

else:
    raise ValueError(f"Unknown BACKEND: {BACKEND!r}")

CFG_PATH.write_text(yaml.dump(cfg, sort_keys=False, allow_unicode=True))

print("\nWrote config.yaml → llm:")
print(cfg["llm"])
print("\nOPENAI_API_KEY in environment:", bool(os.environ.get("OPENAI_API_KEY")))

## 8) Run a natural-language query

In [ ]:
from pathlib import Path

from nlp_sql.pipeline import answer_request

cfg = Path("config.yaml")
result = answer_request(
    "List employees in HR with salary over 100000",
    config_path=cfg,
)

print("SQL:", result.sql)
print("DBs:", result.database_ids_used)
if result.explanation:
    print("Note:", result.explanation)
print("Columns:", result.column_names)
print("Rows:")
for row in result.rows:
    print(row)

## 9) Optional: HTTP API in Colab

Starts Uvicorn with `PYTHONPATH` so `nlp_sql` imports work. Use Colab’s **proxy URL** for `/query` from your browser.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

from google.colab.output import eval_js

ROOT = Path(os.getcwd()).resolve()
env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT / "src")

log_path = ROOT / "uvicorn.log"
log = open(log_path, "w")
proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "nlp_sql.api:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd=str(ROOT),
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
)
time.sleep(2)

print("Uvicorn PID:", proc.pid)
print("Logs:", log_path)
print("Proxy URL (open in a new tab while this runtime is alive):")
print(eval_js("google.colab.kernel.proxyPort(8000)"))